# Scraping

**Situs:** Wikipedia - *List of countries and dependencies by area* (halaman HTML biasa, tanpa API)

## Langkah 1 - Install dan import library

In [1]:
# Fitrya Irfan_202431073
# Langkah 1: install dan import library
!pip install requests beautifulsoup4 pandas

import requests
import csv
import re
import pandas as pd
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser


## Langkah 2 - Cek robots.txt, ambil halaman web, dan cek status_code

In [2]:
# Fitrya Irfan_202431073
# Langkah 2: ambil halaman web dan cek status_code
url = "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_area"
robots_url = "https://en.wikipedia.org/robots.txt"

headers = {
    "User-Agent": "Praktikum-Pemsin-ITPLN/1.0 (tugas kuliah; Fitrya Irfan_202431073)"
}

# Baca robots.txt lalu cek apakah halaman boleh diambil
robots_response = requests.get(robots_url, headers=headers, timeout=30)
print("Alamat robots.txt :", robots_url)
print("Status robots.txt :", robots_response.status_code)

rp = RobotFileParser()
rp.parse(robots_response.text.splitlines())
izin = rp.can_fetch(headers["User-Agent"], url)
print("Boleh di-scrape   :", izin)

if not izin:
    raise Exception("Halaman tidak diizinkan oleh robots.txt")

# Ambil halaman dan cek status_code (harus 200)
response = requests.get(url, headers=headers, timeout=30)
print("Status halaman    :", response.status_code)
assert response.status_code == 200, "status_code bukan 200"


Alamat robots.txt : https://en.wikipedia.org/robots.txt
Status robots.txt : 200
Boleh di-scrape   : True
Status halaman    : 200


## Langkah 3 - Ubah isi halaman menjadi objek BeautifulSoup

In [3]:
# Fitrya Irfan_202431073
# Langkah 3: ubah HTML menjadi objek BeautifulSoup (html.parser)
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.get_text(strip=True))


List of countries and dependencies by area - Wikipedia


## Langkah 4 - Cari semua blok data (find_all) dan tampilkan jumlahnya

In [4]:
# Fitrya Irfan_202431073
# Langkah 4: cari semua blok data. Satu baris tabel (<tr>) = satu blok data
tabel_list = soup.find_all("table", class_="wikitable")
print("Jumlah tabel wikitable :", len(tabel_list))

tabel = tabel_list[0]
semua_tr = tabel.find_all("tr")
blok = [tr for tr in semua_tr if tr.find("td")]   # buang baris header (hanya <th>)

print("Jumlah baris <tr>      :", len(semua_tr))
print("Jumlah blok data       :", len(blok))
print("Memenuhi minimal 200?  :", len(blok) >= 200)


Jumlah tabel wikitable : 1
Jumlah baris <tr>      : 278
Jumlah blok data       : 276
Memenuhi minimal 200?  : True


## Langkah 5 - Ambil tiap atribut dari tiap blok (find, get_text(strip=True)) ke list result

In [5]:
# Fitrya Irfan_202431073
# Langkah 5: ambil atribut tiap blok memakai find, lalu get_text(strip=True), simpan ke list result
def ambil_angka(teks):
    """Ambil angka pertama dari teks (boleh desimal), hasilnya string tanpa koma ribuan."""
    teks = re.sub(r"\[.*?\]", "", teks)               # buang penanda catatan kaki [1]
    cocok = re.search(r"\d[\d,]*\.?\d*", teks)
    return cocok.group().replace(",", "") if cocok else ""

def ambil_td(tr, urutan):
    """Ambil sel <td> ke-urutan (mulai 0) memakai find lalu find_next_sibling."""
    td = tr.find("td")
    for _ in range(urutan):
        td = td.find_next_sibling("td") if td else None
    return td

result = []

for tr in blok:
    for sup in tr.find_all("sup"):                     # buang catatan kaki
        sup.decompose()

    td_nama  = ambil_td(tr, 1)
    td_total = ambil_td(tr, 2)
    td_darat = ambil_td(tr, 3)
    td_air   = ambil_td(tr, 4)
    td_persen = ambil_td(tr, 5)

    if None in (td_nama, td_total, td_darat, td_air, td_persen):   # lewati baris tidak lengkap
        continue

    nama = td_nama.get_text(strip=True)                # contoh: "Greenland(Denmark)"
    negara = re.sub(r"\s*\(.*", "", nama).strip()

    result.append({
        "Country": negara,
        "Total_Area_km2": ambil_angka(td_total.get_text(strip=True)),
        "Land_Area_km2": ambil_angka(td_darat.get_text(strip=True)),
        "Water_Area_km2": ambil_angka(td_air.get_text(strip=True)),
        "Water_Percent": ambil_angka(td_persen.get_text(strip=True)),
        "Status": "Dependensi" if "(" in nama else "Berdaulat",
    })

print("Jumlah data pada list result:", len(result))

Jumlah data pada list result: 276


## Langkah 6 - Tampilkan hasil dengan perulangan for

In [6]:
# Fitrya Irfan_202431073
# Langkah 6: tampilkan hasil scraping dengan perulangan for (10 data pertama)
for data in result[:10]:
    print("Negara      :", data["Country"])
    print("Luas total  :", data["Total_Area_km2"], "km2")
    print("Luas darat  :", data["Land_Area_km2"], "km2")
    print("Luas air    :", data["Water_Area_km2"], "km2")
    print("% air       :", data["Water_Percent"])
    print("Status      :", data["Status"])
    print("-" * 35)


Negara      : Earth
Luas total  : 510072000 km2
Luas darat  : 148940000 km2
Luas air    : 361132000 km2
% air       : 70.8
Status      : Berdaulat
-----------------------------------
Negara      : Russia
Luas total  : 17098246 km2
Luas darat  : 16376870 km2
Luas air    : 721380 km2
% air       : 4.2
Status      : Berdaulat
-----------------------------------
Negara      : Antarctica
Luas total  : 14200000 km2
Luas darat  : 14200000 km2
Luas air    : 0 km2
% air       : 0
Status      : Berdaulat
-----------------------------------
Negara      : Canada
Luas total  : 9984670 km2
Luas darat  : 9093507 km2
Luas air    : 891163 km2
% air       : 8.9
Status      : Berdaulat
-----------------------------------
Negara      : China
Luas total  : 9596960 km2
Luas darat  : 9326410 km2
Luas air    : 270550 km2
% air       : 2.8
Status      : Berdaulat
-----------------------------------
Negara      : United States
Luas total  : 9525067 km2
Luas darat  : 9147593 km2
Luas air    : 377424 km2
% air   

## Langkah 7 - Simpan ke file CSV (csv.DictWriter)

In [7]:
# Fitrya Irfan_202431073
# Langkah 7: simpan ke CSV memakai csv.DictWriter
nama_file = "hasil_scraping_negara.csv"
kolom_csv = ["Country", "Total_Area_km2", "Land_Area_km2",
             "Water_Area_km2", "Water_Percent", "Status"]

with open(nama_file, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=kolom_csv)
    writer.writeheader()
    for data in result:
        writer.writerow(data)

print("Data berhasil disimpan ke", nama_file)


Data berhasil disimpan ke hasil_scraping_negara.csv


## Langkah 8 - Baca file CSV dengan pandas dan tampilkan datanya

In [8]:
# Fitrya Irfan_202431073
# Langkah 8: baca CSV dengan pandas
data = pd.read_csv("hasil_scraping_negara.csv")
print(data)
print("Ukuran dataset:", data.shape)


                         Country  Total_Area_km2  Land_Area_km2  \
0                          Earth    5.100720e+08   1.489400e+08   
1                         Russia    1.709825e+07   1.637687e+07   
2                     Antarctica    1.420000e+07   1.420000e+07   
3                         Canada    9.984670e+06   9.093507e+06   
4                          China    9.596960e+06   9.326410e+06   
..                           ...             ...            ...   
271  Ashmore and Cartier Islands    5.000000e+00   5.000000e+00   
272            Coral Sea Islands    3.000000e+00   3.000000e+00   
273              Spratly Islands    2.000000e+00   2.000000e+00   
274                       Monaco    2.000000e+00   2.000000e+00   
275                 Vatican City    4.400000e-01   4.400000e-01   

     Water_Area_km2  Water_Percent      Status  
0       361132000.0           70.8   Berdaulat  
1          721380.0            4.2   Berdaulat  
2               0.0            0.0   Berdaulat  

## Langkah 9 - Pastikan kolom angka bertipe angka, lalu simpan

In [9]:
# Fitrya Irfan_202431073
# Langkah 9: pastikan kolom angka bertipe angka (int/float)
print(data.dtypes)

kolom_angka = ["Total_Area_km2", "Land_Area_km2", "Water_Area_km2", "Water_Percent"]
for kolom in kolom_angka:
    data[kolom] = pd.to_numeric(data[kolom], errors="coerce")

# Buang baris yang bukan negara/dependensi (Earth, Antarctica) dan wilayah klaim
# tanpa daratan (luas darat = 0), serta baris yang angkanya kosong
data = data[~data["Country"].isin(["Earth", "World", "Antarctica"])]
data = data[data["Land_Area_km2"] > 0]
data = data.dropna(subset=kolom_angka).drop_duplicates(subset="Country")
data = data.reset_index(drop=True)

data.to_csv("hasil_scraping_negara.csv", index=False)   # simpan kembali

print(data.dtypes)
print(data.head())
print("Jumlah data akhir:", len(data))
print(data["Status"].value_counts())


Country            object
Total_Area_km2    float64
Land_Area_km2     float64
Water_Area_km2    float64
Water_Percent     float64
Status             object
dtype: object
Country            object
Total_Area_km2    float64
Land_Area_km2     float64
Water_Area_km2    float64
Water_Percent     float64
Status             object
dtype: object
         Country  Total_Area_km2  Land_Area_km2  Water_Area_km2  \
0         Russia      17098246.0     16376870.0        721380.0   
1         Canada       9984670.0      9093507.0        891163.0   
2          China       9596960.0      9326410.0        270550.0   
3  United States       9525067.0      9147593.0        377424.0   
4         Brazil       8510346.0      8460415.0         55352.0   

   Water_Percent     Status  
0            4.2  Berdaulat  
1            8.9  Berdaulat  
2            2.8  Berdaulat  
3            4.0  Berdaulat  
4            0.6  Berdaulat  
Jumlah data akhir: 254
Berdaulat     203
Dependensi     51
Name: Status, dtyp